# Adversarial Dataset Generation Using BERTAttack

Generate adversarial training data for model robustness improvement.

## Configuration
- **Attack**: BERTAttack (Li et al., 2020)
- **Dataset**: LIAR train split
- **USE Threshold**: 0.8
- **Max Perturbation**: 10% of words

In [1]:
# Imports
import os, gc, json, random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, set_seed
import textattack
from textattack.models.wrappers import HuggingFaceModelWrapper
from textattack.attack_recipes import BERTAttackLi2020
from textattack.constraints.overlap import MaxWordsPerturbed
from textattack.datasets import Dataset as TADataset
from textattack import Attacker
from textattack.attack_args import AttackArgs

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

## 1. Load Dataset and Build Text

In [ ]:
# Load LIAR
ds = load_dataset('ucsbnlp/liar')
print({k: len(ds[k]) for k in ds.keys()})

# Label configuration
LABELS = ['false', 'half-true', 'mostly-true', 'true', 'barely-true', 'pants-fire']
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}

def safe_str(x):
    if x is None: return ''
    if isinstance(x, (list, tuple)):
        return ', '.join([str(i) for i in x if i is not None])
    return str(x).strip()

def build_text(example):
    statement = safe_str(example.get('statement', ''))
    subject = safe_str(example.get('subjects', example.get('subject', '')))
    context = safe_str(example.get('context', ''))
    parts = [statement]
    if subject: parts.append(f'SUBJECT: {subject}')
    if context: parts.append(f'CONTEXT: {context}')
    example['text'] = ' [SEP] '.join(parts)
    y = example.get('label')
    if isinstance(y, str):
        example['label_id'] = label2id[y.strip().lower()]
    else:
        example['label_id'] = int(y)
    return example

ds = ds.map(build_text)
print(f"Example: {ds['train'][0]['text'][:150]}...")


KeyboardInterrupt



## 2. Load Model

In [ ]:
MODEL_DIR = 'baseline_42'
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, model_max_length=MAX_LENGTH, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.config.id2label = id2label
model.config.label2id = label2id

print(f'Loaded: {MODEL_DIR} | num_labels: {model.config.num_labels}')

## 3. Configure BERTAttack

In [ ]:
# GPU setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()
torch.set_grad_enabled(False)

# Build attack
model_wrapper = HuggingFaceModelWrapper(model, tokenizer)
attack = BERTAttackLi2020.build(model_wrapper)

# Configure
USE_THRESHOLD = 0.8
MAX_PCT_WORDS = 0.10
MAX_CANDIDATES = 5

for c in attack.constraints:
    if c.__class__.__name__ == 'UniversalSentenceEncoder':
        c.threshold = USE_THRESHOLD

attack.transformation.max_length = MAX_LENGTH
attack.transformation.max_candidates = MAX_CANDIDATES

attack.constraints = [c for c in attack.constraints 
                      if c.__class__.__name__ not in ('MaxPercentWordsPerturbed', 'MaxWordsPerturbed')]
attack.constraints.append(MaxWordsPerturbed(max_percent=MAX_PCT_WORDS))

print(f'Attack configured: USE={USE_THRESHOLD}, max_words={MAX_PCT_WORDS*100}%')

In [ ]:
# GPU setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()
torch.set_grad_enabled(False)

# Build attack
model_wrapper = HuggingFaceModelWrapper(model, tokenizer)
attack = BERTAttackLi2020.build(model_wrapper)

# Configure
USE_THRESHOLD = 0.8
MAX_PCT_WORDS = 0.10
MAX_CANDIDATES = 5

for c in attack.constraints:
    if c.__class__.__name__ == 'UniversalSentenceEncoder':
        c.threshold = USE_THRESHOLD

attack.transformation.max_length = MAX_LENGTH
attack.transformation.max_candidates = MAX_CANDIDATES

attack.constraints = [c for c in attack.constraints 
                      if c.__class__.__name__ not in ('MaxPercentWordsPerturbed', 'MaxWordsPerturbed')]
attack.constraints.append(MaxWordsPerturbed(max_percent=MAX_PCT_WORDS))

print(f'Attack configured: USE={USE_THRESHOLD}, max_words={MAX_PCT_WORDS*100}%')

## 4. Generate Adversarial Examples

In [ ]:
# Setup
SPLIT = 'train'
N_EXAMPLES = 10269
BATCH_SIZE = 128
OUT_DIR = 'ADVERSARIAL_TRAINING_DATA_42'
os.makedirs(OUT_DIR, exist_ok=True)

STATE_PATH = os.path.join(OUT_DIR, f'progress_{SPLIT}.json')
JSONL_PATH = os.path.join(OUT_DIR, f'adv_success_{SPLIT}.jsonl')

# Prepare data
split_ds = ds[SPLIT].select(range(min(N_EXAMPLES, len(ds[SPLIT]))))
ta_pairs = [(ex['text'], ex['label_id']) for ex in split_ds]
print(f'Generating adversarial data for {len(ta_pairs)} examples')

In [ ]:
# Checkpoint functions
def save_state(next_index, success_count):
    tmp = STATE_PATH + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump({'next_index': next_index, 'success_count': success_count}, f)
    os.replace(tmp, STATE_PATH)

def append_success(rows):
    with open(JSONL_PATH, 'a', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

# Load progress
start_idx, success_count = 0, 0
if os.path.exists(STATE_PATH):
    with open(STATE_PATH, 'r', encoding='utf-8') as f:
        st = json.load(f)
    start_idx = int(st.get('next_index', 0))
    success_count = int(st.get('success_count', 0))
    print(f'Resuming from {start_idx} (success: {success_count})')

In [ ]:
# Generate adversarial examples
total = len(ta_pairs)
print(f'Starting generation: {total} total, batch size {BATCH_SIZE}\n')

for i in range(start_idx, total, BATCH_SIZE):
    j = min(i + BATCH_SIZE, total)
    batch_pairs = ta_pairs[i:j]
    batch_data = TADataset(batch_pairs)
    
    attack_args = AttackArgs(
        num_examples=len(batch_data),
        query_budget=80,
        random_seed=SEED,
        disable_stdout=True,
        log_to_csv=None
    )
    
    attacker = Attacker(attack, batch_data, attack_args)
    batch_results = attacker.attack_dataset()
    
    rows = []
    for k, r in enumerate(batch_results):
        if r.__class__.__name__ == 'SuccessfulAttackResult':
            adv_text = r.perturbed_result.attacked_text.text
            label = int(r.original_result.ground_truth_output)
            rows.append({'text': adv_text, 'label': label, 'orig_index': i + k})
    
    success_count += len(rows)
    if rows:
        append_success(rows)
    save_state(next_index=j, success_count=success_count)
    
    print(f'Batch [{i}:{j}] done | successful total: {success_count}')
    
    del attacker, batch_results, batch_data
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'\nComplete! {success_count} adversarial examples saved to {JSONL_PATH}')

## 5. Save Dataset

In [ ]:
# Load and save as HF Dataset
rows = []
with open(JSONL_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

adv_df = pd.DataFrame(rows)
adv_hf = Dataset.from_pandas(adv_df.reset_index(drop=True))

adv_hf.save_to_disk(os.path.join(OUT_DIR, f'adv_only_hf_{SPLIT}'))
adv_df.to_csv(os.path.join(OUT_DIR, f'adv_only_{SPLIT}.csv'), index=False)

print(f'Saved {len(adv_df)} adversarial examples')
print(f'Label distribution:')
for label_id, count in adv_df['label'].value_counts().sort_index().items():
    print(f'  {id2label[label_id]}: {count}')